In [1]:
f = open("/home/c24ma/src/sstal/benchmarks/lexaz/countdown/a.out", 'rb')

In [2]:
from elftools.elf.elffile import ELFFile
elf = ELFFile(f)
stackmaps_section = elf.get_section_by_name('__LLVM_StackMaps')

In [3]:
stackmaps_section = elf.get_section_by_name('.llvm_stackmaps')
data = stackmaps_section.data()

In [4]:
from elftools.dwarf.descriptions import _REG_NAMES_x64
import struct

def int_to_string(num):
    result = ''
    while num > 0:
        result = chr(num % 256) + result
        num //= 256
    return result

def parse_stackmaps(data):
    # Parse the header
    header_fmt = 'BBH'
    header_size = struct.calcsize(header_fmt)
    stack_map_version, reserved1, reserved2 = struct.unpack(header_fmt, data[:header_size])
    print(f"Stack Map Version: {stack_map_version}")
    print(f"Reserved1: {reserved1}")
    print(f"Reserved2: {reserved2}")

    # Set the initial offset past the header
    offset = header_size
    
    # Parse NumFunctions, NumConstants, NumRecords
    num_fmt = 'III'
    num_size = struct.calcsize(num_fmt)
    num_functions, num_constants, num_records = struct.unpack_from(num_fmt, data, offset)
    offset += num_size
    
    print(f"NumFunctions: {num_functions}, NumConstants: {num_constants}, NumRecords: {num_records}")

    # Parse StkSizeRecord
    stk_size_fmt = 'QQQ'
    stk_size_len = struct.calcsize(stk_size_fmt)
    print("Function Stack Sizes:")
    for _ in range(num_functions):
        address, stack_size, record_count = struct.unpack_from(stk_size_fmt, data, offset)
        offset += stk_size_len
        print(f"  Address: {address:#x}, Stack Size: {stack_size}, Record Count: {record_count}")

    # Parse Constants
    const_fmt = 'Q'
    const_size = struct.calcsize(const_fmt)
    print("Constants:")
    for _ in range(num_constants):
        large_constant = struct.unpack_from(const_fmt, data, offset)
        offset += const_size
        print(f"  Large Constant: {large_constant[0]}")

    # Parse StkMapRecords
    print("Stack Map Records:")
    for _ in range(num_records):
        rec_header_fmt = 'QIH'
        rec_header_size = struct.calcsize(rec_header_fmt)
        patch_point_id, instruction_offset, reserved = struct.unpack_from(rec_header_fmt, data, offset)
        offset += rec_header_size
        
        num_locations, = struct.unpack_from('H', data, offset)
        offset += 2

        print(f"  Patch Point ID: {int_to_string(patch_point_id)}, Instruction Offset: {instruction_offset}, NumLocations: {num_locations}")

        # Loop through each location
        loc_fmt = 'BBHHHi'
        loc_size = struct.calcsize(loc_fmt)
        for _ in range(num_locations):
            type, reserved_loc, location_size, dwarf_regnum, reserved_dwarf, offset_smallconst = struct.unpack_from(loc_fmt, data, offset)
            assert(reserved_loc == 0)
            assert(reserved_dwarf == 0)
            offset += loc_size
            print(f"    Type: {type}, Location Size: {location_size}, Dwarf RegNum: {_REG_NAMES_x64[dwarf_regnum]}, Offset/SmallConstant: {offset_smallconst}")

        # Padding for alignment to 8 bytes
        alignment = offset % 8
        if alignment != 0:
            offset += 8 - alignment

        num_live_outs, = struct.unpack_from('H', data, offset)
        offset += 2
        print(f"  NumLiveOuts: {num_live_outs}")

        live_out_fmt = 'HBB'
        live_out_size = struct.calcsize(live_out_fmt)
        for _ in range(num_live_outs):
            dwarf_regnum, reserved_live, size_in_bytes = struct.unpack_from(live_out_fmt, data, offset)
            offset += live_out_size
            print(f"    LiveOut - Dwarf RegNum: {dwarf_regnum}, Size in Bytes: {size_in_bytes}")

        # Additional padding for alignment to 8 bytes, if required
        alignment = offset % 8
        if alignment != 0:
            offset += 8 - alignment

parse_stackmaps(data)

Stack Map Version: 3
Reserved1: 0
Reserved2: 0
NumFunctions: 3, NumConstants: 0, NumRecords: 4
Function Stack Sizes:
  Address: 0x3570, Stack Size: 168, Record Count: 1
  Address: 0x36f0, Stack Size: 152, Record Count: 1
  Address: 0x37b0, Stack Size: 72, Record Count: 2
Constants:
Stack Map Records:
  Patch Point ID: main, Instruction Offset: 279, NumLocations: 1
    Type: 2, Location Size: 8, Dwarf RegNum: rsp, Offset/SmallConstant: 8
  NumLiveOuts: 0
  Patch Point ID: lifted_2, Instruction Offset: 145, NumLocations: 1
    Type: 2, Location Size: 8, Dwarf RegNum: rsp, Offset/SmallConstant: 0
  NumLiveOuts: 0
  Patch Point ID: lifted_1, Instruction Offset: 32, NumLocations: 1
    Type: 2, Location Size: 8, Dwarf RegNum: rsp, Offset/SmallConstant: 8
  NumLiveOuts: 0
  Patch Point ID: lifted_1, Instruction Offset: 71, NumLocations: 1
    Type: 2, Location Size: 8, Dwarf RegNum: rsp, Offset/SmallConstant: 8
  NumLiveOuts: 0


In [33]:
def djb2(s):
    hash = 5381
    for c in s:
        hash = ((hash << 5) + hash + ord(c)) % 4294967291  # hash * 33 + c
    return hash

s = "__runProduct_lifted_3__"
print(djb2(s))

4200373063


In [40]:
from elftools.dwarf.callframe import ZERO
from elftools.dwarf.descriptions import _REG_NAMES_x64
cfi_entries = elf.get_dwarf_info().EH_CFI_entries()
for cfi in cfi_entries:
    if type(cfi) is ZERO:
        continue
    table = cfi.get_decoded().table
    table.sort(key=lambda x: x["pc"])
    for entry in table:
        if entry["cfa"].reg:
            print({"pc": hex(entry["pc"]), "cfa": (_REG_NAMES_x64[entry["cfa"].reg], entry["cfa"].offset)})

{'pc': '0x0', 'cfa': ('rsp', 8)}
{'pc': '0x2230', 'cfa': ('rsp', 8)}
{'pc': '0x2234', 'cfa': ('rsp', 8)}
{'pc': '0x2020', 'cfa': ('rsp', 16)}
{'pc': '0x2026', 'cfa': ('rsp', 24)}
{'pc': '0x2220', 'cfa': ('rsp', 8)}
{'pc': '0x2320', 'cfa': ('rsp', 8)}
{'pc': '0x2321', 'cfa': ('rsp', 16)}
{'pc': '0x2350', 'cfa': ('rsp', 8)}
{'pc': '0x2359', 'cfa': ('rsp', 16)}
{'pc': '0x2380', 'cfa': ('rsp', 8)}
{'pc': '0x2381', 'cfa': ('rsp', 16)}
{'pc': '0x239a', 'cfa': ('rsp', 8)}
{'pc': '0x23a0', 'cfa': ('rsp', 8)}
{'pc': '0x23a1', 'cfa': ('rsp', 16)}
{'pc': '0x23bb', 'cfa': ('rsp', 8)}
{'pc': '0x23c0', 'cfa': ('rsp', 8)}
{'pc': '0x23c2', 'cfa': ('rsp', 16)}
{'pc': '0x23c3', 'cfa': ('rsp', 24)}
{'pc': '0x23c4', 'cfa': ('rsp', 32)}
{'pc': '0x23e4', 'cfa': ('rsp', 24)}
{'pc': '0x23e5', 'cfa': ('rsp', 16)}
{'pc': '0x23e7', 'cfa': ('rsp', 8)}
{'pc': '0x23e8', 'cfa': ('rsp', 32)}
{'pc': '0x2400', 'cfa': ('rsp', 8)}
{'pc': '0x2402', 'cfa': ('rsp', 16)}
{'pc': '0x2403', 'cfa': ('rsp', 24)}
{'pc': '0x2404', 